# Realtor Modeling Notebook

This notebook prepares modeling-ready housing data and tests several linear regression approaches for the BUSA 695 capstone project. It starts from the raw dataset, engineers features, encodes categories, trains multiple regression setups, evaluates model performance, and saves a modeling dataset for later use.

**Datasets loaded:** `realtor-data.csv` and `realtor_master.csv`

**Main steps:**
- Load the source data.
- Engineer a `zip3` feature and remove high-cardinality columns.
- Handle missing values, reduce data size, and encode categories.
- Train and evaluate multiple linear regression model variations.
- Save the final modeling dataset for reuse.

**Outputs created:** `realtor_modeling.csv` and notebook-based model evaluation results.

**Capstone connection:** This notebook builds the structured modeling dataset and baseline regression results that support comparison with later random forest work.
            

In [1]:
# ===============================
# 1. IMPORT LIBRARIES
# ===============================
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score



c:\Users\A1990\anaconda3\envs\dev\lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


## Main Modeling Data Pipeline

The first part of this notebook creates a modeling-ready dataset from the raw realtor file through feature engineering, cleaning, encoding, and a baseline linear regression workflow.
            

In [2]:
# ===============================
# 2. LOAD DATA (ONLY ONCE)
# ===============================
first_df = pd.read_csv("realtor-data.csv")

# Quick check
print(first_df.shape)
first_df.head()

(2226382, 12)


,brokered_by,status,price,bed,bath,acre_lot,street,city,state,zip_code,house_size,prev_sold_date
0,103378.0,for_sale,105000.0,3.0,2.0,0.12,1962661.0,Adjuntas,Puerto Rico,601.0,920.0,NaN
1,52707.0,for_sale,80000.0,4.0,2.0,0.08,1902874.0,Adjuntas,Puerto Rico,601.0,1527.0,NaN
2,103379.0,for_sale,67000.0,2.0,1.0,0.15,1404990.0,Juana Diaz,Puerto Rico,795.0,748.0,NaN
3,31239.0,for_sale,145000.0,4.0,2.0,0.10,1947675.0,Ponce,Puerto Rico,731.0,1800.0,NaN
4,34632.0,for_sale,65000.0,6.0,2.0,0.05,331151.0,Mayaguez,Puerto Rico,680.0,NaN,NaN


In [3]:
# ===============================
# 3. FEATURE ENGINEERING (ZIP3)
# MUST COME BEFORE DROPPING zip_code
# ===============================
first_df["zip3"] = first_df["zip_code"].astype(str).str[:3]

In [4]:
# ===============================
# 4. DROP USELESS / HIGH CARDINALITY COLUMNS
# ===============================
first_df = first_df.drop(columns=[
    "city",            # too many categories
    "zip_code",        # replaced by zip3
    "street",          # unique values
    "brokered_by",     # ID, not useful
    "prev_sold_date"   # requires advanced processing
])

In [5]:
# ===============================
# 5. HANDLE MISSING VALUES
# ===============================

# Drop rows where target is missing
first_df = first_df.dropna(subset=["price"])

# Fill numerical columns with MEDIAN
first_df["bed"] = first_df["bed"].fillna(first_df["bed"].median())
first_df["bath"] = first_df["bath"].fillna(first_df["bath"].median())
first_df["acre_lot"] = first_df["acre_lot"].fillna(first_df["acre_lot"].median())
first_df["house_size"] = first_df["house_size"].fillna(first_df["house_size"].median())

In [6]:
# ===============================
# 6. REDUCE DATA SIZE (VERY IMPORTANT)
# ===============================
first_df = first_df.sample(100000, random_state=42)

In [7]:
# ===============================
# 7. LIMIT ZIP3 CATEGORIES (PREVENT MEMORY CRASH)
# ===============================
top_zip3 = first_df["zip3"].value_counts().nlargest(100).index

first_df["zip3"] = first_df["zip3"].where(
    first_df["zip3"].isin(top_zip3),
    "other"
)

In [8]:
# ===============================
# 8. ENCODE CATEGORICAL VARIABLES
# ===============================
first_df = pd.get_dummies(
    first_df,
    columns=["state", "zip3", "status"],
    drop_first=True
)

In [9]:
# ===============================
# 9. FINAL CHECK (NO STRINGS)
# ===============================
print(first_df.select_dtypes(include=['object']).columns)

Index([], dtype='object')


In [10]:
first_df = first_df[first_df["price"] > 0]

## Prepare Features and Train the First Linear Regression Model

This section defines the target and predictors, splits the data, and evaluates the first model built from the processed dataset.
            

In [11]:
# Define the modeling target and predictors after all preprocessing steps are complete.
# Avoid division errors
first_df = first_df[first_df["bed"] > 0]
first_df = first_df[first_df["house_size"] > 0]

# Feature engineering
first_df["price_per_sqft"] = first_df["price"] / first_df["house_size"]
first_df["bath_per_bed"] = first_df["bath"] / first_df["bed"]
first_df["rooms_total"] = first_df["bed"] + first_df["bath"]
first_df["lot_to_size"] = first_df["acre_lot"] / first_df["house_size"]

In [12]:
# ===============================
# 10. CREATE X AND y
# ===============================
X = first_df.drop("price", axis=1)
y = np.log(first_df["price"])  # log transform improves model

In [13]:
# ===============================
# 11. TRAIN / TEST SPLIT
# ===============================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [14]:
# Train a comparison model using the cleaned master dataset with a different location setup.
# ===============================
# 12. TRAIN MODEL
# ===============================
model_1 = LinearRegression()
model_1.fit(X_train, y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [15]:
#creating a model in LinearRegression -- train -- predict -- evaluate 
model_1 = LinearRegression()

# Step 1: Train the model
model_1.fit(X_train, y_train) # type: ignore

# Step 2: Create predictions (THIS WAS MISSING)
y_pred = model_1.predict(X_test) # type: ignore

# Step 3: Evaluate
r2 = r2_score(y_test, y_pred)
print("R²:", r2)

R²: 0.46844043250222445


## Compare Additional Linear Regression Setups

These cells test alternative feature sets using the cleaned master file to compare how location variables affect performance.
            

In [16]:
second_df = pd.read_csv("realtor_master.csv")
second_df

C:\Users\A1990\AppData\Local\Temp\ipykernel_21904\599021063.py:1: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  second_df = pd.read_csv("realtor_master.csv")


,price,bed,bath,acre_lot,city,state,zip_code,house_size,status_for_sale,status_ready_to_build,status_sold
0,105000.0,3.0,2.0,0.12,Adjuntas,Puerto Rico,601.0,920.0,True,False,False
1,80000.0,4.0,2.0,0.08,Adjuntas,Puerto Rico,601.0,1527.0,True,False,False
2,67000.0,2.0,1.0,0.15,Juana Diaz,Puerto Rico,795.0,748.0,True,False,False
3,145000.0,4.0,2.0,0.10,Ponce,Puerto Rico,731.0,1800.0,True,False,False
4,65000.0,6.0,2.0,0.05,Mayaguez,Puerto Rico,680.0,1760.0,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...
2224836,359900.0,4.0,2.0,0.33,Richland,Washington,99354.0,3600.0,False,False,True
2224837,350000.0,3.0,2.0,0.10,Richland,Washington,99354.0,1616.0,False,False,True
2224838,440000.0,6.0,3.0,0.50,Richland,Washington,99354.0,3200.0,False,False,True
2224839,179900.0,2.0,1.0,0.09,Richland,Washington,99354.0,933.0,False,False,True


In [17]:
#removing variables we do not need for the model 
second_df = second_df.drop(columns=["city", "zip_code"])
second_df

,price,bed,bath,acre_lot,state,house_size,status_for_sale,status_ready_to_build,status_sold
0,105000.0,3.0,2.0,0.12,Puerto Rico,920.0,True,False,False
1,80000.0,4.0,2.0,0.08,Puerto Rico,1527.0,True,False,False
2,67000.0,2.0,1.0,0.15,Puerto Rico,748.0,True,False,False
3,145000.0,4.0,2.0,0.10,Puerto Rico,1800.0,True,False,False
4,65000.0,6.0,2.0,0.05,Puerto Rico,1760.0,True,False,False
...,...,...,...,...,...,...,...,...,...
2224836,359900.0,4.0,2.0,0.33,Washington,3600.0,False,False,True
2224837,350000.0,3.0,2.0,0.10,Washington,1616.0,False,False,True
2224838,440000.0,6.0,3.0,0.50,Washington,3200.0,False,False,True
2224839,179900.0,2.0,1.0,0.09,Washington,933.0,False,False,True


In [18]:
#Encoding state to 1 and 0
second_df = pd.get_dummies(second_df, columns=["state"])

In [19]:
# Build a third comparison model that keeps state and capped zip code categories.
#Defining X and the target Y 

X = second_df.drop("price", axis = 1) # Assolating price 
y = second_df["price"] # we want to predict price (target)

In [20]:
#Spliting data -- training thge model on 80% of the data and testing it with the remining (20%) unsean data

X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=42)

In [21]:
#creating a model in LinearRegression -- train -- predict -- evaluate 
model_2 = LinearRegression()

# Step 1: Train the model
model_2.fit(X_train, y_train) 

# Step 2: Create predictions 
y_pred = model_2.predict(X_test) 

# Step 3: Evaluate
r2 = r2_score(y_test, y_pred)
print("R²:", r2)

R²: 0.06465104141921774


In [22]:
third_df = pd.read_csv("realtor_master.csv")
third_df

C:\Users\A1990\AppData\Local\Temp\ipykernel_21904\1667790980.py:1: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  third_df = pd.read_csv("realtor_master.csv")


,price,bed,bath,acre_lot,city,state,zip_code,house_size,status_for_sale,status_ready_to_build,status_sold
0,105000.0,3.0,2.0,0.12,Adjuntas,Puerto Rico,601.0,920.0,True,False,False
1,80000.0,4.0,2.0,0.08,Adjuntas,Puerto Rico,601.0,1527.0,True,False,False
2,67000.0,2.0,1.0,0.15,Juana Diaz,Puerto Rico,795.0,748.0,True,False,False
3,145000.0,4.0,2.0,0.10,Ponce,Puerto Rico,731.0,1800.0,True,False,False
4,65000.0,6.0,2.0,0.05,Mayaguez,Puerto Rico,680.0,1760.0,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...
2224836,359900.0,4.0,2.0,0.33,Richland,Washington,99354.0,3600.0,False,False,True
2224837,350000.0,3.0,2.0,0.10,Richland,Washington,99354.0,1616.0,False,False,True
2224838,440000.0,6.0,3.0,0.50,Richland,Washington,99354.0,3200.0,False,False,True
2224839,179900.0,2.0,1.0,0.09,Richland,Washington,99354.0,933.0,False,False,True


In [23]:
#Loading daset with state and zip_code only 
third_df = third_df.drop(columns=["city"])

In [24]:
#Loading only top zip_code categories to a maz of 100
top_zips = third_df["zip_code"].value_counts().nlargest(100).index

third_df["zip_code"] = third_df["zip_code"].apply(
    lambda x: x if x in top_zips else "Other"
)

In [25]:
#enconding variables state and zip_code to 1 and 0
third_df = pd.get_dummies(third_df, columns=["state", "zip_code"])

In [26]:
#defining x and y 
X = third_df.drop("price", axis=1)
y = third_df["price"]

In [27]:
#Spliting the data to train on 80% of the data and test on the reiming 20%
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [28]:
#Creating model_3 and fitting our data to the model 
model_3 = LinearRegression()
model_3.fit(X_train, y_train)

y_pred = model_3.predict(X_test)

In [29]:
#Evaluating the output of the model_3
r2 = r2_score(y_test, y_pred)
print("Model 3 R²:", r2)

Model 3 R²: 0.0721178099913784


## Save the Modeling Dataset

The final step exports the processed modeling table so other project notebooks can reuse the same prepared features.
            

In [30]:
# Export the final prepared modeling dataset for later project notebooks.
modeling_df = first_df.copy()

modeling_df.to_csv("realtor_modeling.csv", index=False)

In [32]:
modeling_df.head()

,price,bed,bath,acre_lot,house_size,state_Alaska,state_Arizona,state_Arkansas,state_California,state_Colorado,...,zip3_982,zip3_983,zip3_986,zip3_other,status_ready_to_build,status_sold,price_per_sqft,bath_per_bed,rooms_total,lot_to_size
749154,395000.0,4.0,4.0,0.35,2884.0,False,False,False,False,False,...,False,False,False,True,False,False,136.962552,1.0,8.0,0.000121
664508,370000.0,3.0,3.0,2.80,2700.0,False,False,False,False,False,...,False,False,False,True,False,False,137.037037,1.0,6.0,0.001037
2073318,1550000.0,5.0,3.0,0.12,3000.0,False,False,False,True,False,...,False,False,False,True,False,True,516.666667,0.6,8.0,0.000040
781031,88000.0,2.0,2.0,0.19,1069.0,False,False,False,False,False,...,False,False,False,True,False,False,82.319925,1.0,4.0,0.000178
1327900,1595000.0,2.0,1.0,0.06,1665.0,False,False,False,True,False,...,False,False,False,True,False,False,957.957958,0.5,3.0,0.000036
